# NanoInspect: tier-2 fine-tuning result and a capacity-aware tier-1 threshold

Uses only the saved outputs of `nanoinspect_tier2ft.ipynb` (tier 2 fine-tuned) and `artifacts/results_baseline/` (before fine-tuning). CPU only.

1. **Keep or revert the tier-2 fine-tune?** The rule, fixed before training: keep it only if the cascade's cost per 1,000 parts beats the baseline's $415.24.
2. **How many parts should go to tier 2?** (saved as an optional *capacity mode*; the default *throughput mode* keeps tier 2 for doubtful parts only, so every outcome, including automatic rejects, stays in play) After fine-tuning, tier 2 is the strongest model, so routing more parts to it pays off, but only as many as it can serve. The threshold comes from **validation** parts and **measured serving capacity**, never from test labels:
   - capacity share = 0.8 × (tier-2 parts/min at its best concurrency) ÷ (line parts/min)
   - T_LO = the tier-1 P(defect) above which that share of held-out *validation* good parts fall

In [1]:
import sys, os, json
sys.path.insert(0, os.getcwd())
import numpy as np, pandas as pd
from IPython.display import display
from nanoinspect import policy
from nanoinspect.config import RESULTS_DIR, ARTIFACTS
B = json.loads((ARTIFACTS / "results_baseline" / "benchmark_summary.json").read_text())
N = json.loads((RESULTS_DIR / "benchmark_summary.json").read_text())
qb = {r["config"]: r for r in B["model_quality"]}; qn = {r["config"]: r for r in N["model_quality"]}
rows = []
for k in qn:
    for key in ["roc_auc", "recall", "false_positive_rate", "accuracy", "defect_type_acc", "location_acc"]:
        pass
    rows.append({"model": k, **{f"{m} before": qb.get(k, {}).get(m) for m in ["roc_auc", "recall", "false_positive_rate", "defect_type_acc", "location_acc"]},
                 **{f"{m} after": qn[k].get(m) for m in ["roc_auc", "recall", "false_positive_rate", "defect_type_acc", "location_acc"]}})
display(pd.DataFrame(rows).round(3))
cb = B["escalation_policy"]["held_out"]["NanoInspect cascade"]["cost_per_1000_usd"]; cn = N["escalation_policy"]["held_out"]["NanoInspect cascade"]["cost_per_1000_usd"]
KEEP = cn < cb
print(f"Cascade cost per 1,000 parts: before ${cb:.2f}, after ${cn:.2f} -> {'KEEP the tier-2 fine-tune' if KEEP else 'REVERT'}")
t2b = max(B["serving"]["tier2"], key=lambda r: r["requests_per_s"]); t2n = max(N["serving"]["tier2"], key=lambda r: r["requests_per_s"])
print(f"Price of fine-tuning tier 2: served in BF16 (NVFP4 cannot carry the adapter): {t2b['requests_per_s']} -> {t2n['requests_per_s']} parts/s, "
      f"{B['S5_energy']['tier2']['joules_per_item']:.0f} -> {N['S5_energy']['tier2']['joules_per_item']:.0f} J per tier-2 call")

,model,roc_auc before,recall before,false_positive_rate before,defect_type_acc before,location_acc before,roc_auc after,recall after,false_positive_rate after,defect_type_acc after,location_acc after
0,Tier 1: Qwen2.5-VL-7B + LoRA (fine-tuned on Nano),0.962,0.844,0.054,0.742,0.701,0.962,0.844,0.056,0.736,0.706
1,Baseline: Qwen2.5-VL-7B zero-shot,0.836,0.210,0.019,0.523,0.500,0.837,0.208,0.017,0.527,0.481
2,Tier 2: Qwen3.8-27B with good reference,0.956,0.892,0.111,0.540,0.656,0.988,0.943,0.047,0.781,0.808
3,"Baseline: Qwen3.8-27B zero-shot, single image",0.913,0.785,0.116,0.510,0.632,0.924,0.747,0.077,0.513,0.643
4,Baseline: ResNet-18 classifier (non-LLM),0.960,0.928,0.148,NaN,NaN,0.960,0.928,0.148,NaN,NaN
5,Baseline: training-free delta vs good parts (n...,0.892,0.625,0.041,NaN,NaN,0.892,0.625,0.041,NaN,NaN


Cascade cost per 1,000 parts: before $415.24, after $230.07 -> KEEP the tier-2 fine-tune
Price of fine-tuning tier 2: served in BF16 (NVFP4 cannot carry the adapter): 0.95 -> 0.46 parts/s, 51 -> 122 J per tier-2 call


In [2]:
LINE_PPM = 60
cap_ppm = max(r["requests_per_s"] for r in N["serving"]["tier2"]) * 60
share = min(1.0, 0.8 * cap_ppm / LINE_PPM)
val = pd.read_csv(RESULTS_DIR / "llm_t1_val.csv")
T_LO_CAP = float(np.quantile(val.p_defective, 1 - share))
print(f"tier-2 capacity {cap_ppm:.1f} parts/min at a {LINE_PPM} parts/min line -> may send {share:.1%} of parts -> T_LO = {T_LO_CAP:.4f} "
      f"({(val.p_defective >= T_LO_CAP).mean():.1%} of validation good parts go to tier 2)")

t1 = pd.read_csv(RESULTS_DIR / "llm_t1_ft.csv"); t2 = pd.read_csv(RESULTS_DIR / "llm_t2_ref.csv").set_index("path")
res = pd.read_csv(RESULTS_DIR / "vision_predictions.csv"); res = res[res.variant == "C_real_defects"].set_index("path")
z27 = pd.read_csv(RESULTS_DIR / "llm_t2_zero.csv").set_index("path")
df = t1[["path", "category", "label"]].copy(); df["score"] = t1.p_defective; df["verdict"] = df.path.map(t2.verdict)
df["t2_flag"] = df.verdict == "defective"; df["resnet_flag"] = df.path.map(res.score) >= 0.5; df["t2zero_flag"] = df.path.map(z27.verdict) == "defective"
out = {}
for name, tlo in [("threshold from validation 95th percentile (before)", N.get("t_lo", 0.5)), ("capacity-aware threshold (new)", T_LO_CAP)]:
    d = df.copy(); d["review_at"] = tlo
    m = np.random.default_rng(0).random(len(d)) < 0.5
    t = policy.policy_table(policy.fit_likelihoods(d[m]), policy.DEFAULT_COSTS, policy.DEFAULT_DEFECT_RATE)
    out[f"NanoInspect cascade, {name}"] = {**policy.simulate(d[~m], t, policy.DEFAULT_DEFECT_RATE, policy.DEFAULT_COSTS, 0.05), "t_lo": tlo}
cmp = pd.DataFrame(out).T
display(cmp.round(3))

tier-2 capacity 27.6 parts/min at a 60 parts/min line -> may send 36.8% of parts -> T_LO = 0.0474 (36.7% of validation good parts go to tier 2)


,escapes_per_1000,false_rejects_per_1000,vlm_calls_per_1000,human_reviews_per_1000,cost_per_1000_usd,t_lo
"NanoInspect cascade, threshold from validation 95th percentile (before)",3.871,0.0,524.187,73.001,230.074,0.047
"NanoInspect cascade, capacity-aware threshold (new)",3.871,0.0,524.187,73.001,230.074,0.047


In [3]:
# Save BOTH policies. Throughput mode (default, T_LO from validation) keeps tier 2 for doubtful parts only;
# capacity mode uses the capacity-aware T_LO. The app can switch between them on the Policy page.
EB = {"Tier 2 alone (27B with reference decides)": "t2_flag", "Baseline: 27B zero-shot alone": "t2zero_flag", "Baseline: ResNet-18 alone (non-LLM)": "resnet_flag"}
T_LO_DEFAULT = 0.5
d0 = df.copy(); d0["review_at"] = T_LO_DEFAULT
table0, held_out0, _ = policy.fit_and_save(d0, costs=policy.DEFAULT_COSTS, defect_rate=policy.DEFAULT_DEFECT_RATE, audit_rate=0.05, t_lo=T_LO_DEFAULT, extra_baselines=EB)
d = df.copy(); d["review_at"] = T_LO_CAP
table, held_out, lik = policy.fit_and_save(d, costs=policy.DEFAULT_COSTS, defect_rate=policy.DEFAULT_DEFECT_RATE, audit_rate=0.05, t_lo=T_LO_CAP,
                                           extra_baselines=EB, path=policy.CAPACITY_PATH)
print("Throughput mode (default):"); display(table0.drop(columns=["meaning"]).round(4)); display(held_out0.round(3).sort_values("cost_per_1000_usd"))
print("Capacity mode:")
display(table.drop(columns=["meaning"]).round(4)); display(held_out.round(3).sort_values("cost_per_1000_usd"))
N["tier2_finetune_decision"] = {"rule": "keep if cascade cost per 1,000 parts < baseline", "baseline_cost": cb, "fine_tuned_cost": cn, "keep": bool(KEEP)}
N["t_lo_capacity_aware"] = {"t_lo": T_LO_CAP, "line_ppm": LINE_PPM, "tier2_capacity_ppm": cap_ppm, "share_to_tier2": share}
N["t_lo"] = T_LO_DEFAULT
N["escalation_policy"]["held_out"] = held_out0.round(4).to_dict("index")
N["escalation_policy"]["table"] = table0[["bucket", "p_defect", "action"]].round(5).to_dict("records")
N["escalation_policy"]["t_lo"] = T_LO_DEFAULT
N["escalation_policy_capacity_mode"] = {"t_lo": T_LO_CAP, "held_out": held_out.round(4).to_dict("index"),
                                        "table": table[["bucket", "p_defect", "action"]].round(5).to_dict("records")}
(RESULTS_DIR / "benchmark_summary.json").write_text(json.dumps(N, indent=1, default=str))
print("saved policy and summary")

Throughput mode (default):


,bucket,p_defect,expected_cost_accept_usd,expected_cost_reject_usd,human_review_usd,action,evidence_defect_parts,evidence_good_parts
0,fast_accept,0.0091,0.4555,1.9818,0.5,accept,51,207
1,agree_good,0.0017,0.0843,1.9966,0.5,accept,8,195
2,agree_defect,0.8219,41.0956,0.3562,0.5,reject,250,1
3,t1_flag_t2_good,0.0393,1.9628,1.9215,0.5,escalate_to_human,9,8
4,t1_good_t2_defect,0.1107,5.5345,1.7786,0.5,escalate_to_human,43,12
5,t2_unusable,0.0355,1.7735,1.9291,0.5,escalate_to_human,0,0


,escapes_per_1000,false_rejects_per_1000,vlm_calls_per_1000,human_reviews_per_1000,cost_per_1000_usd
Tier 2 alone (27B with reference decides),2.978,34.064,0.000,0.000,217.030
NanoInspect cascade,7.226,11.355,151.478,55.832,411.910
Baseline: ResNet-18 alone (non-LLM),3.762,128.685,0.000,0.000,445.458
Human inspects every part,0.000,0.000,0.000,1000.000,500.000
Tier 1 alone (fine-tuned 7B decides),7.524,64.343,0.000,0.000,504.861
Baseline: 27B zero-shot alone,12.539,75.697,0.000,0.000,778.354


Capacity mode:


,bucket,p_defect,expected_cost_accept_usd,expected_cost_reject_usd,human_review_usd,action,evidence_defect_parts,evidence_good_parts
0,fast_accept,0.0017,0.0827,1.9967,0.5,accept,4,110
1,agree_good,0.0007,0.0340,1.9986,0.5,accept,1,107
2,agree_defect,0.4931,24.6560,1.0138,0.5,escalate_to_human,290,10
3,t1_flag_t2_good,0.0064,0.3202,1.9872,0.5,accept,16,96
4,t1_good_t2_defect,0.0355,1.7735,1.9291,0.5,escalate_to_human,3,3
5,t2_unusable,0.0355,1.7735,1.9291,0.5,escalate_to_human,0,0


,escapes_per_1000,false_rejects_per_1000,vlm_calls_per_1000,human_reviews_per_1000,cost_per_1000_usd
Tier 2 alone (27B with reference decides),2.978,34.064,0.000,0.000,217.030
NanoInspect cascade,3.871,0.000,524.187,73.001,230.074
Baseline: ResNet-18 alone (non-LLM),3.762,128.685,0.000,0.000,445.458
Human inspects every part,0.000,0.000,0.000,1000.000,500.000
Tier 1 alone (fine-tuned 7B decides),7.524,64.343,0.000,0.000,504.861
Baseline: 27B zero-shot alone,12.539,75.697,0.000,0.000,778.354


saved policy and summary
